In [6]:
# Project paths on your two systems
from pathlib import Path


PROJECT_PATHS = [
    Path(r"D:\Projects\done\eth_bot\eth_bot_v2"),
    Path(r"D:\Sachin Chakrawarti\eth_bot\eth_bot_v2"),
]

# Automatically select the project path that exists
for project_path in PROJECT_PATHS:
    if project_path.exists():
        sys.path.insert(0, str(project_path))
        break
else:
    raise FileNotFoundError(
        "eth_bot_v2 project directory was not found on this system."
    )


DB = Path(r"D:\Sachin Chakrawarti\eth_bot\eth_bot_v2") / "database" / "market.db"

In [7]:
# --- setup --------------------------------------------------------
import sys





import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path

DB = Path(r"D:\Projects\done\eth_bot\eth_bot_v2\database\market.db")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (12, 4)

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from analysis.loader import load_ohlcv
from analysis.indicators import add_all
from analysis import stats


In [8]:
# --- load ---------------------------------------------------------
oc = sql("SELECT * FROM option_chain ORDER BY expiry, strike")
oc["expiry_date"] = pd.to_datetime(oc.expiry, unit="ms")
print(oc.shape)
oc.head(20)

OperationalError: unable to open database file

In [ ]:
# --- IV by strike for the nearest expiry --------------------------
near = oc[oc.expiry == oc.expiry.min()]
near[near["kind"] == "C"].set_index("strike")["iv"].plot(title="IV smile (calls)")
plt.tight_layout()
plt.show()

In [ ]:
# --- put/call open interest ---------------------------------------
piv = oc.pivot_table(index="strike", columns="kind", values="oi", aggfunc="sum")
piv["pc_ratio"] = piv["P"] / piv["C"]
piv.tail(20)